In [0]:
 #══════════════════════════════════════
# INGESTION — physical_lojas
# Squad 3 — Batch Ecommerce
# ══════════════════════════════════════

# Acesso ao config e utils

%run "/Workspace/Repos/luizhpdatasci@gmail.com/merca-data-platform/Squad3/luiz-portacio/config/00_config.ipynb"
%run "/Workspace/Repos/luizhpdatasci@gmail.com/merca-data-platform/Squad3/luiz-portacio/utils/00_utils.ipynb"

In [0]:
# Listar arquivos

arquivos = listar_arquivos(adls_client, container)

In [0]:
# Ler arquivo

df_lojas = ler_csv(adls_client, container, "physical_lojas.csv")
print(f"✅ physical_lojas lido! Shape: {df_lojas.shape}")


In [0]:
# Análise Exploratória

print("=" * 50)
print("📊 ANÁLISE EXPLORATÓRIA — physical_lojas")
print("=" * 50)

print(f"\n📐 Shape: {df_lojas.shape}")
print(f"   {df_lojas.shape[0]} linhas | {df_lojas.shape[1]} colunas")

print("\n📋 Colunas e tipos:")
print(df_lojas.dtypes)

print("\n❓ Nulos por coluna:")
print(df_lojas.isnull().sum())

print(f"\n🔁 Duplicatas: {df_lojas.duplicated().sum()}")

print("\n📈 Estatísticas:")
df_lojas.describe()

In [0]:
# Primeiras Linhas

print("👀 Primeiras 10 linhas:")
df_lojas.head(10)

In [0]:
# Tratamentos

print("=" * 50)
print("🔧 TRATAMENTOS — physical_lojas")
print("=" * 50)

# ══════════════════════════════════════
# 1. CNPJ — converter para string
# ══════════════════════════════════════
# CNPJ não pode ser int64 — perde zeros à esquerda
df_lojas['cnpj'] = df_lojas['cnpj'].astype(str).str.zfill(14)
print("✅ cnpj convertido para string com zeros à esquerda")

# ══════════════════════════════════════
# 2. ID — converter para string
# ══════════════════════════════════════
df_lojas['id_loja'] = df_lojas['id_loja'].astype(str)
print("✅ id_loja convertido para string")

# ══════════════════════════════════════
# 3. TEXTO — padronizar
# ══════════════════════════════════════
df_lojas['nome_loja']   = df_lojas['nome_loja'].str.strip().str.upper()
df_lojas['cidade_loja'] = df_lojas['cidade_loja'].str.strip().str.upper()
df_lojas['estado_loja'] = df_lojas['estado_loja'].str.strip().str.upper()
print("✅ Colunas texto padronizadas para maiúsculo")

# ══════════════════════════════════════
# 4. ESTADO — verificar valores válidos
# ══════════════════════════════════════
estados_validos = [
    'AC','AL','AP','AM','BA','CE','DF','ES','GO',
    'MA','MT','MS','MG','PA','PB','PR','PE','PI',
    'RJ','RN','RS','RO','RR','SC','SP','SE','TO'
]
estados_invalidos = df_lojas[
    ~df_lojas['estado_loja'].isin(estados_validos)
]['estado_loja'].unique()

if len(estados_invalidos) > 0:
    print(f"⚠️  Estados inválidos encontrados: {estados_invalidos}")
else:
    print("✅ Todos os estados são válidos")

# ══════════════════════════════════════
# 5. PESO_VENDAS — verificar negativos
# ══════════════════════════════════════
negativos = (df_lojas['peso_vendas'] < 0).sum()
if negativos > 0:
    df_lojas['peso_vendas'] = df_lojas['peso_vendas'].abs()
    print(f"✅ {negativos} valores negativos em peso_vendas corrigidos")
else:
    print("✅ peso_vendas sem valores negativos")

# ══════════════════════════════════════
# RESULTADO FINAL
# ══════════════════════════════════════
print(f"\n📐 Shape final: {df_lojas.shape}")
print(f"\n👀 Amostra final:")
df_lojas.head()

In [0]:
# Salvar no SQL Server

salvar_tabela(df_lojas, nome_tabela="physical_lojas")

In [0]:
# Verificar Dados Salvos

df_verificacao = consultar_tabela("physical_lojas")
print(f"✅ Verificação concluída!")
print(f"   Linhas no banco: {len(df_verificacao)}")
df_verificacao.head()